# Course 6 — Feature Selection II

Shrinkage methods: Ridge, Lasso, Elastic Net. Instead of picking
predictors in or out, every coefficient is pulled toward zero — softly
for Ridge, sharply for Lasso.

**Sections**
1. Ridge regression (0:00–0:30)
2. Lasso and sparsity (0:30–1:00)
3. Elastic Net and the end-to-end pipeline (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold

hitters = load_dataset('hitters').dropna(subset=['Salary']).reset_index(drop=True)
hitters = pd.get_dummies(hitters, columns=['League', 'Division', 'NewLeague'],
                          drop_first=True, dtype=float)
y = hitters['Salary'].to_numpy()
X = hitters.drop(columns=['Salary']).to_numpy()
feat_names = hitters.drop(columns=['Salary']).columns.tolist()
print(X.shape)


## 1. Ridge regression

Ridge minimizes $\sum (y_i - \hat y_i)^2 + \alpha \sum \beta_j^2$.
The `α` knob trades training fit (α → 0) for shrinkage (α → ∞).

In [ ]:
alphas = np.logspace(-2, 4, 80)
coef_paths = np.empty((len(alphas), len(feat_names)))
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)
for i, a in enumerate(alphas):
    coef_paths[i] = Ridge(alpha=a).fit(Xs, y).coef_

fig, ax = plt.subplots(figsize=(7, 4.5))
for j, name in enumerate(feat_names):
    ax.plot(alphas, coef_paths[:, j], label=name)
ax.set_xscale('log'); ax.set_xlabel(r'$\alpha$'); ax.set_ylabel('coefficient')
ax.set_title('Ridge coefficient paths on Hitters'); plt.show()


### Pick α with CV

In [ ]:
pipe = Pipeline([('s', StandardScaler()), ('r', Ridge())])
grid = GridSearchCV(pipe, {'r__alpha': np.logspace(-2, 4, 25)},
                    cv=KFold(5, shuffle=True, random_state=0),
                    scoring='neg_mean_squared_error', n_jobs=-1)
grid.fit(X, y)
print(f'best alpha = {grid.best_params_["r__alpha"]:.3f}')
print(f'CV MSE     = {-grid.best_score_:.0f}')


## 2. Lasso and sparsity

Lasso minimizes $\sum (y_i - \hat y_i)^2 + \alpha \sum |\beta_j|$.
The L1 penalty creates a *corner* at zero, so some coefficients get
set exactly to zero.

In [ ]:
alphas = np.logspace(-1, 3, 60)
nz_count = []; coef_paths_l = np.empty((len(alphas), len(feat_names)))
for i, a in enumerate(alphas):
    m = Lasso(alpha=a, max_iter=20000).fit(Xs, y)
    coef_paths_l[i] = m.coef_
    nz_count.append(int((m.coef_ != 0).sum()))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for j in range(len(feat_names)):
    axes[0].plot(alphas, coef_paths_l[:, j])
axes[0].set_xscale('log'); axes[0].set_xlabel(r'$\alpha$')
axes[0].set_ylabel('coefficient'); axes[0].set_title('Lasso paths')
axes[1].plot(alphas, nz_count, marker='o')
axes[1].set_xscale('log'); axes[1].set_xlabel(r'$\alpha$')
axes[1].set_ylabel('# non-zero coefficients'); axes[1].set_title('Sparsity')
plt.tight_layout(); plt.show()


In [ ]:
pipe = Pipeline([('s', StandardScaler()), ('l', Lasso(max_iter=20000))])
grid = GridSearchCV(pipe, {'l__alpha': np.logspace(-1, 3, 30)},
                    cv=KFold(5, shuffle=True, random_state=0),
                    scoring='neg_mean_squared_error', n_jobs=-1)
grid.fit(X, y)
best = grid.best_estimator_.named_steps['l']
active = [n for n, c in zip(feat_names, best.coef_) if c != 0]
print(f'best alpha = {grid.best_params_["l__alpha"]:.3f}')
print(f'survivors  = {active}')


## 3. Elastic Net and the end-to-end pipeline

Elastic Net mixes the L1 and L2 penalties. `l1_ratio = 1` is Lasso,
`l1_ratio = 0` is Ridge. Anywhere in between gives Lasso-style sparsity
with Ridge-style stability when groups of features are correlated.

In [ ]:
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('poly', PolynomialFeatures(2, interaction_only=True, include_bias=False)),
    ('lasso', Lasso(max_iter=20000)),
])
grid = GridSearchCV(pipe, {'lasso__alpha': np.logspace(-1, 2, 12)},
                    cv=KFold(5, shuffle=True, random_state=0),
                    scoring='neg_mean_squared_error', n_jobs=-1)
grid.fit(X, y)
best = grid.best_estimator_.named_steps['lasso']
n_active = int((best.coef_ != 0).sum())
print(f'best alpha = {grid.best_params_["lasso__alpha"]:.3f}')
print(f'CV MSE     = {-grid.best_score_:.0f}')
print(f'{n_active} non-zero coefs out of {best.coef_.size} (poly features)')


### Recap
- Ridge shrinks every coefficient toward zero — none ever reaches zero.
- Lasso shrinks *and* zeros some coefficients out — it's a feature
  selector and a regularizer in one.
- Elastic Net is the compromise for correlated-feature situations.
- Wrap preprocessing + estimator in a `Pipeline` and grid-search the
  regularization strength via CV.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Fit a Lasso to predict `fare` on `titanic`. Standardize
first. Tune `alpha` with 5-fold CV. Print the survivors and the chosen α.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd, numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd, numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold
df = load_dataset('titanic').dropna(subset=['fare']).reset_index(drop=True)
df = pd.get_dummies(df[['fare','pclass','sex','age','sibsp','parch','embarked','class']].fillna(df.median(numeric_only=True)),
                     columns=['sex','embarked','class'], drop_first=True, dtype=float)
y = df['fare'].to_numpy(); X = df.drop(columns=['fare'])
pipe = Pipeline([('s', StandardScaler()), ('l', Lasso(max_iter=20000))])
grid = GridSearchCV(pipe, {'l__alpha': np.logspace(-2, 2, 25)},
                    cv=KFold(5, shuffle=True, random_state=0),
                    scoring='neg_mean_squared_error', n_jobs=-1).fit(X, y)
best = grid.best_estimator_.named_steps['l']
print(f'alpha = {grid.best_params_["l__alpha"]:.3f}')
for name, c in zip(X.columns, best.coef_):
    if c != 0: print(f'  keep {name:18s} coef = {c:+.3f}')


---

## Exercise 2

**Task 2.** Re-run with Ridge instead. Are any coefficients exactly
zero? Compare CV MSE between Ridge and Lasso.

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
from sklearn.linear_model import Ridge
pipe_r = Pipeline([('s', StandardScaler()), ('r', Ridge())])
grid_r = GridSearchCV(pipe_r, {'r__alpha': np.logspace(-2, 4, 25)},
                       cv=KFold(5, shuffle=True, random_state=0),
                       scoring='neg_mean_squared_error', n_jobs=-1).fit(X, y)
ridge = grid_r.best_estimator_.named_steps['r']
n_zero = int((ridge.coef_ == 0).sum())
print(f'Ridge: alpha={grid_r.best_params_["r__alpha"]:.2f}  CV-MSE={-grid_r.best_score_:.0f}  exact-zero coefs={n_zero}')
print(f'Lasso: alpha={grid.best_params_["l__alpha"]:.2f}  CV-MSE={-grid.best_score_:.0f}  exact-zero coefs={int((best.coef_==0).sum())}')


---

## Exercise 3

**Task 3.** Does Lasso pick the same features as forward stepwise from
Course 5? List both and report the overlap size.

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
# forward stepwise to convergence
remaining = list(X.columns); chosen = []; mses = []
while remaining:
    best_col, best_mse = None, float('inf')
    for col in remaining:
        s = -cross_val_score(LinearRegression(), X[chosen+[col]].to_numpy(), y,
                              cv=5, scoring='neg_mean_squared_error').mean()
        if s < best_mse: best_mse, best_col = s, col
    chosen.append(best_col); remaining.remove(best_col); mses.append(best_mse)
fwd = set(chosen[: mses.index(min(mses)) + 1])
lasso = set(n for n, c in zip(X.columns, best.coef_) if c != 0)
print(f'forward stepwise picked {len(fwd)} features: {sorted(fwd)}')
print(f'lasso picked            {len(lasso)} features: {sorted(lasso)}')
print(f'overlap                  {len(fwd & lasso)} features: {sorted(fwd & lasso)}')
